Simulation d'un réseau de neurones 3x3

In [217]:
#!pip install pyswarm -q
#!pip install bayesian-optimization -q
#!pip install mealpy -q
#!pip install scikit-optimize -q

In [218]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
from sklearn.metrics import f1_score

# Pour la recherche bayésienne
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

In [219]:
class ThreeByThreeNN(nn.Module):
    def __init__(self):
        super(ThreeByThreeNN, self).__init__()
        self.fc1 = nn.Linear(3, 3)  # couche 1 : 3 → 3
        self.fc2 = nn.Linear(3, 3)  # couche 2 : 3 → 3
        self.fc3 = nn.Linear(3, 3)  # couche 3 : 3 → 3

    def forward(self, x):
        x = F.relu(self.fc1(x))  # activation ReLU après couche 1
        x = F.relu(self.fc2(x))  # activation ReLU après couche 2
        x = self.fc3(x)          # pas d'activation finale (à adapter selon la tâche)
        return x


Masque pour le dropout

In [220]:
p1 = 0.05 #Note : c'est la proba de dropout donc proba de masquer le neurone zij
p2 = 0.005
p3 = 0.05

class ThreeByThreeNN_MCdropout(nn.Module):
    def __init__(self, model, p1=0.05, p2=0.005, p3=0.05):
        super().__init__()
        self.model = model
        self.p1 = p1
        self.p2 = p2
        self.p3 = p3
    
    def dropout_mask(self, x, p):
        mask = (torch.rand_like(x) > p).float() / (1 - p) ##garde x1 si proba >p1, on normalise pour que l'espérance reste constante
        return mask

    def forward(self, x):
        # Couche 1 + dropout manuel
        x1 = F.relu(self.model.fc1(x))
        mask1 = self.dropout_mask(x1, self.p1)
        x1_d = x1 * mask1 #vecteur x1 auquel on a appliqué le masque, correspond à Wi

        # Couche 2 + dropout manuel
        x2 = F.relu(self.model.fc2(x1_d))
        mask2 = self.dropout_mask(x2, self.p2)
        x2_d = x2 * mask2

        # Couche 3 + dropout manuel
        x3 = self.model.fc3(x2_d)
        mask3 = self.dropout_mask(x3, self.p3)
        x3_d = x3 * mask3

        # Affichage des masques
        #print("Masque couche 1:\n", mask1)
        #print("Masque couche 2:\n", mask2)
        #print("Masque couche 3:\n", mask3)

        return x3_d


Données

In [221]:
model_orig = ThreeByThreeNN()
model = ThreeByThreeNN_MCdropout(model_orig, p1=0.05, p2=0.005, p3=0.05)

In [222]:
# Vecteur d'entrée X et sortie attendue Y (batch de taille 1)
X = torch.randn(1, 3)  # entrée shape [1, 3]
Y = model_orig(X)
Y_hat = model(X)
N = X.shape[0]

print("X : ensemble des valeurs d'entrée :", X)
print("Y : ensemble des valeurs de sortie :", Y)
print("Y_hat : ensemble des valeurs de sortie avec dropout", Y_hat)
print("N : nombre total de données :", N)

X : ensemble des valeurs d'entrée : tensor([[-0.8614,  0.5951,  0.5140]])
Y : ensemble des valeurs de sortie : tensor([[-0.5464, -0.0750, -0.5242]], grad_fn=<AddmmBackward0>)
Y_hat : ensemble des valeurs de sortie avec dropout tensor([[-0.5765, -0.0787, -0.5526]], grad_fn=<MulBackward0>)
N : nombre total de données : 1


MC Dropout : inférence

In [223]:
T = 1000 #nb de forward passes
outputs = []
model.train()  # dropout activé même en inférence

with torch.no_grad():
    for _ in range(T):
        out = model(X)
        outputs.append(out.unsqueeze(0))

outputs = torch.cat(outputs, dim=0)
mean_pred = outputs.mean(dim=0)  # moyenne sur les T passes, MC estimate
uncertainty = outputs.var(dim=0)  # variance = incertitude estimée

In [224]:
print("MC estimate:", mean_pred)
print("MC Dropout Uncertainty :", uncertainty)

MC estimate: tensor([[-0.5391, -0.0773, -0.5256]])
MC Dropout Uncertainty : tensor([[0.0191, 0.0004, 0.0143]])


Métriques d'incertitude

Note : j'ai mis peu d'itérations sinon le code tourne trop longtemps mais serait plus précis

Note : plus tard, prendre en compte le bruit pour l'UA ;
Ici, qu'une seule entrée X donc c'est soit 0 soit 1, à tester avec plus de valeurs

In [225]:
def compute_ua(mean_probs, Y_true, threshold=0.7):#seuil détermine si c'est certain ou pas, cf p.12 article Enhancing
    pred_class = mean_probs.argmax(dim=1)
    confidence = mean_probs.max(dim=1).values

    correct = pred_class == Y_true
    certain = confidence >= threshold

    TC = ((correct) & (certain)).sum().item()
    TU = ((~correct) & (~certain)).sum().item()
    total = len(Y_true)

    ua = (TC + TU) / total
    return ua 

In [226]:
def compute_ece(probs, labels, n_bins=10):
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(labels)
    ece = torch.zeros(1)
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        prop_in_bin = in_bin.float().mean()
        if prop_in_bin.item() > 0:
            acc_in_bin = accuracies[in_bin].float().mean()
            avg_conf = confidences[in_bin].mean()
            ece += torch.abs(avg_conf - acc_in_bin) * prop_in_bin
    return ece

In [227]:
def run_dropout_trial(p1, p2, p3, X, Y, T=1000):
    model = ThreeByThreeNN_MCdropout(ThreeByThreeNN(), p1=p1, p2=p2, p3=p3)
    model.train()

    outputs = [model(X).unsqueeze(0).detach() for _ in range(T)]
    outputs = torch.cat(outputs, dim=0)  # [T, batch, n_classes]

    # Cibles
    Y_class = torch.argmax(Y, dim=1)
    Y_one_hot = F.one_hot(Y_class, num_classes=3).float()

    # NLL sur la prédiction moyenne 
    mean_pred = outputs.mean(dim=0)
    log_probs = F.log_softmax(mean_pred, dim=1)
    nll = F.nll_loss(log_probs, Y_class)

    # Brier point par point puis moyenne sur T passes
    brier_list = []
    with torch.no_grad():
        for t in range(T):
            probs_t = torch.softmax(outputs[t], dim=1)
            brier_per_point = torch.sum((probs_t - Y_one_hot) ** 2, dim=1)  # [batch]
            brier_list.append(brier_per_point.mean().item())
    brier = sum(brier_list) / len(brier_list)

    # ECE et UA sur la prédiction moyenne
    mean_probs = torch.softmax(mean_pred, dim=1)
    ece = compute_ece(mean_probs, Y_class)
    ua = compute_ua(mean_probs, Y_class)

    return nll.item(), brier, ece.item(), mean_probs, Y_one_hot, ua, (p1, p2, p3)


In [228]:
nll, brier, ece, mean_probs, Y_one_hot, ua, (p1, p2, p3) = run_dropout_trial(p1, p2, p3, X, Y)

print("Negative Log-likelihood", nll)
print("Moyenne des probas", mean_probs)
print("Classe de Y", Y_one_hot)
print("Brier score moyen", brier)
print("Expected Calibration Error", ece)
print("Uncertain Accuracy", ua)

Negative Log-likelihood 0.7454577684402466
Moyenne des probas tensor([[0.1578, 0.4745, 0.3677]])
Classe de Y tensor([[0., 1., 0.]])
Brier score moyen 0.4391705850958824
Expected Calibration Error 0.5254829525947571
Uncertain Accuracy 0.0


Evoquer divergence KL à minimiser (Gal et Ghahramani)

Modèle final avec les meilleures probas

In [229]:
# Utilisation des meilleures probabilités
model_final = ThreeByThreeNN_MCdropout(ThreeByThreeNN())
model_final.train()

T = 1000
outputs = []
with torch.no_grad():
    for _ in range(T):
        outputs.append(model_final(X).unsqueeze(0))
outputs = torch.cat(outputs, dim=0)
mean_pred = outputs.mean(dim=0)

print("MC Dropout output (mean logits):", mean_pred)

MC Dropout output (mean logits): tensor([[ 0.3341, -0.3266,  0.0961]])


In [230]:
Y_final = model_final(X)
distance = torch.norm(Y_hat - Y, p=2)
print("Distance Euclidienne :", distance.item())

Distance Euclidienne : 0.04156266525387764


In [231]:
#Remarque : la variance c'est pour la régression, sinon c'est la predictive entropy pour la classification

uncertainty = outputs.var(dim=0) #variance empirique
print("MC Dropout Uncertainty:", uncertainty) 

MC Dropout Uncertainty: tensor([[0.0056, 0.0064, 0.0010]])


Critères d'incertitude

In [232]:
def compute_ua(mean_probs, Y_true, threshold=0.7):#seuil détermine si c'est certain ou pas, cf p.12 article Enhancing
    pred_class = mean_probs.argmax(dim=1)
    confidence = mean_probs.max(dim=1).values

    correct = pred_class == Y_true
    certain = confidence >= threshold

    TC = ((correct) & (certain)).sum().item()
    TU = ((~correct) & (~certain)).sum().item()
    total = len(Y_true)

    ua = (TC + TU) / total
    return ua 

In [233]:
def compute_usen(mean_probs, Y_true, threshold=0.7):
    pred_class = mean_probs.argmax(dim=1)
    confidence = mean_probs.max(dim=1).values

    correct = pred_class == Y_true
    certain = confidence >= threshold

    FC = ((~correct) & (certain)).sum().item()
    TU = ((~correct) & (~certain)).sum().item()

    usen = TU / (FC + TU)
    return usen

In [234]:
def compute_up(mean_probs, Y_true, threshold=0.7):
    pred_class = mean_probs.argmax(dim=1)
    confidence = mean_probs.max(dim=1).values

    correct = pred_class == Y_true
    certain = confidence >= threshold

    FU = ((correct) & (~certain)).sum().item()
    TU = ((~correct) & (~certain)).sum().item()

    usen = TU / (TU + FU)
    return usen

F1 Score


In [235]:
def mc_dropout_f1_ratio(model, X, Y_true, T=1000):
    model.train()
    all_probs = []
    y_true_np = Y_true.argmax(dim=1).cpu().numpy()

    f1_per_pass = []

    with torch.no_grad():
        for _ in range(T):
            logits = model(X)
            probs = F.softmax(logits, dim=1)
            all_probs.append(probs.unsqueeze(0))
            
            # prédiction de cette passe
            preds_t = probs.argmax(dim=1).cpu().numpy()
            f1_t = f1_score(y_true_np, preds_t, average="macro")
            f1_per_pass.append(f1_t)

    all_probs = torch.cat(all_probs, dim=0)  # shape [T, batch, n_classes]
    
    # --- F1_MC = moyenne des F1 par passe
    f1_mc = sum(f1_per_pass) / len(f1_per_pass)
    
    # --- F1_Ybar = F1 de la prédiction issue de la moyenne des probas
    mean_probs = all_probs.mean(dim=0)
    preds_mean = mean_probs.argmax(dim=1).cpu().numpy()
    f1_ybar = f1_score(y_true_np, preds_mean, average="macro")
    
    # --- Ratio
    ratio = f1_mc / f1_ybar if f1_ybar != 0 else float('nan')
    
    return f1_mc, f1_ybar, ratio

# Exemple d'utilisation
f1_mc, f1_ybar, ratio = mc_dropout_f1_ratio(model, X, Y_one_hot, T=1000)
print("F1_MC (moyenne par passe) :", f1_mc)
print("F1_Ybar (prédiction moyenne) :", f1_ybar)
print("Ratio F1_MC / F1_Ybar :", ratio)


F1_MC (moyenne par passe) : 0.889
F1_Ybar (prédiction moyenne) : 1.0
Ratio F1_MC / F1_Ybar : 0.889


Fonction Kappa de Cohen

In [236]:
from sklearn.metrics import cohen_kappa_score
import numpy as np
import torch

def cohen_kappa_dropout(model, X, device="cpu"):
    model.eval()
    X = X.to(device)

    # Première passe dropout
    with torch.no_grad():
        logits1 = model(X)
        pred1 = torch.argmax(logits1, dim=1).cpu().numpy()

    # Deuxième passe dropout
    with torch.no_grad():
        logits2 = model(X)
        pred2 = torch.argmax(logits2, dim=1).cpu().numpy()

    # Cas où toutes les prédictions sont identiques
    if len(set(pred1)) == 1 and len(set(pred2)) == 1 and pred1[0] == pred2[0]:
        return 1.0  # Accord parfait

    return cohen_kappa_score(pred1, pred2)


In [237]:
kappa_val = cohen_kappa_dropout(model, X)
print(f"Cohen's κ entre deux passes dropout: {kappa_val:.4f}")


Cohen's κ entre deux passes dropout: 1.0000
